In [2]:
import time

def task(name):
    print(f"{name} started")
    time.sleep(2)
    print(f"{name} finished")

task("Task A")
task("Task B")

Task A started
Task A finished
Task B started
Task B finished


In [3]:
import asyncio

async def task(name):
    print(f"{name} started")
    await asyncio.sleep(2)
    print(f"{name} finished")

async def print_task(name):
    print(f"{name} printed")
    await asyncio.sleep(2)
    print(f"{name} finished")

async def main():
    await task("Task A")
    await print_task("Task B")

await main()
# asyncio.run(main())


Task A started
Task A finished
Task B printed
Task B finished


In [ ]:
await pauses the current async function until an async operation finishes,
without blocking the event loop.

In [5]:
import asyncio

async def task(name):
    print(f"{name} started")
    await asyncio.sleep(2)
    print(f"{name} finished")

async def main():
    task1 = asyncio.create_task(task("Task A"))
    task2 = asyncio.create_task(task("Task B"))

    await task1
    await task2

await main()
# asyncio.run(main())


Task A started
Task B started
Task A finished
Task B finished


In [ ]:
| Line              | Meaning                |
| ----------------- | ---------------------- |
| `create_task()`   | Start task immediately |
| `await`           | Wait for result        |
| `asyncio.sleep()` | Non-blocking wait      |
| Event loop        | Switches tasks         |


In [8]:
import asyncio

async def download(file):
    print(f"Downloading {file}")
    await asyncio.sleep(1)
    print(f"Downloaded {file}")

async def main():
    files = ["file1", "file2", "file3"]

    tasks = []
    for file in files:
        tasks.append(asyncio.create_task(download(file)))

    await asyncio.gather(*tasks)

# asyncio.run(main())
await main()

[<Task pending name='Task-15' coro=<download() running at C:\Users\Umesh Samal\AppData\Local\Temp\ipykernel_41036\4116212473.py:3>>, <Task pending name='Task-16' coro=<download() running at C:\Users\Umesh Samal\AppData\Local\Temp\ipykernel_41036\4116212473.py:3>>, <Task pending name='Task-17' coro=<download() running at C:\Users\Umesh Samal\AppData\Local\Temp\ipykernel_41036\4116212473.py:3>>]
Downloaded file1
Downloaded file2
Downloaded file3


In [ ]:
create_task() schedules a coroutine to run concurrently instead of waiting for it immediately.

In [13]:
import asyncio
import aiohttp

async def fetch():
    async with aiohttp.ClientSession() as session:
        async with session.get("https://httpbin.org/delay/2") as response:
            print("Status:", response.status)
            text = await response.json()
            print("Response received")

#asyncio.run(fetch())
await fetch()

Status: 200
Response received


In [16]:
import asyncio
import aiohttp

async def fetch():
    async with aiohttp.ClientSession() as session:
        async with session.get("") as response:
            print("status:", response.status)
            text = await response.json()

In [19]:
import asyncio
import aiohttp
from typing import List, Dict

async def check_service(session: aiohttp.ClientSession, url: str, timeout: int) -> str:
    try:
        async with session.get(url, timeout=timeout) as response:
            if response.status == 200:
                return "HEALTHY"
            return f"UNHEALTHY (Status: {response.status})"
    except asyncio.TimeoutError:
        return "DOWN (Timeout)"
    except aiohttp.ClientError as e:
        return f"DOWN ({str(e)})"

async def async_health_check(services: List[str],timeout: int = 3) -> Dict[str, str]:
    results = {}
    async with aiohttp.ClientSession() as session:
        tasks = {
            service: asyncio.create_task(
                check_service(session, service, timeout)
            )
            for service in services
        }
        for service, task in tasks.items():
            results[service] = await task
    return results


In [20]:
if __name__ == "__main__":
    services = [
        "https://google.com",
        "http://localhost:8080/health",
        "http://invalid-service"
    ]
 #   results = asyncio.run(async_health_check(services))
    results = await async_health_check(services)
    for service, status in results.items():
        print(f"{service} → {status}")

https://google.com → HEALTHY
http://localhost:8080/health → DOWN (Cannot connect to host localhost:8080 ssl:default [Multiple exceptions: [Errno 10061] Connect call failed ('::1', 8080, 0, 0), [Errno 10061] Connect call failed ('127.0.0.1', 8080)])
http://invalid-service → DOWN (Cannot connect to host invalid-service:80 ssl:default [getaddrinfo failed])


In [21]:
#  Multiple Requests WITHOUT async (Slow)
import requests
import time

urls = [
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/2"
]

start = time.time()

for url in urls:
    requests.get(url)

print("Time:", time.time() - start)


Time: 7.934664964675903


In [31]:
import asyncio
import aiohttp

async def check_service(session : aiohttp.ClientSession,url:str,timeout):
    try:
        async with session.get(url,timeout=timeout) as response:
            if response.status == 200:
                return "HEALTHY"
            return f"UNHEALTHY (Status: {response.status})"
    except Exception as e:
        print(e)

async def solve(services:List[str],timeout: int = 3):
    async with aiohttp.ClientSession() as session:
        request = {}
        tasks = { service : asyncio.create_task(
            check_service(session, service, timeout)
            )
            
            for service in services}
        for service, task in tasks.items():
            results[service] = await task
    return results

In [33]:
urls = [
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/2"
]
res = await solve(urls)
for service, status in res.items():
    print(f"{service} → {status}")



https://google.com → HEALTHY
http://localhost:8080/health → DOWN (Cannot connect to host localhost:8080 ssl:default [Multiple exceptions: [Errno 10061] Connect call failed ('::1', 8080, 0, 0), [Errno 10061] Connect call failed ('127.0.0.1', 8080)])
http://invalid-service → DOWN (Cannot connect to host invalid-service:80 ssl:default [getaddrinfo failed])
https://httpbin.org/delay/2 → None


In [ ]:
await means “wait here until finished”

In [ ]:
1️⃣ What is asyncio and why do we use it?

Answer:
asyncio is a Python library used to write asynchronous, non-blocking code.
We use it to handle many I/O tasks at the same time (HTTP calls, DB, file I/O) efficiently.

👉 Best for I/O-bound tasks, not CPU-bound.

In [ ]:
2️⃣ What is a coroutine?

A coroutine is a function defined with async def that can pause (await) and resume later.

In [ ]:
| `def`            | `async def`              |
| ---------------- | ------------------------ |
| Normal function  | Coroutine                |
| Runs immediately | Returns coroutine object |
| Blocking         | Non-blocking             |


In [ ]:
✅ What does await do?

await waits for the response/result of an async operation

It waits without blocking the event loop.

In [ ]:
5️⃣ What is an event loop?

Answer:
The event loop is the engine that:

Runs coroutines
Schedules tasks
Handles I/O events

In [ ]:
8️⃣ What is asyncio.run()?

Answer:
It:

Creates an event loop
Runs the coroutine
Closes the loop

In [ ]:
9️⃣ Why asyncio.run() fails in Jupyter / FastAPI?

Answer:
Because an event loop already exists.

✔️ Use await directly instead.

In [ ]:
🔟 What is non-blocking I/O?

Answer:
Operations that do not stop execution while waiting for I/O (network, disk).

In [ ]:
1️⃣1️⃣ What is asyncio.create_task()?

Answer:
It schedules a coroutine to run concurrently in the event loop.

In [ ]:
1️⃣3️⃣ What is asyncio.gather()?

Answer:
Runs multiple coroutines together and waits for all results.

    await asyncio.gather(c1(), c2())


In [ ]:
gather() → simple parallel execution

create_task() → more control (cancel, monitor)

In [ ]:
2️⃣1️⃣ Why aiohttp over requests?

Answer:
requests is blocking.
aiohttp is non-blocking and async-friendly.

In [ ]:
2️⃣2️⃣ Multiple async HTTP calls?

Answer:
Use create_task() + gather()

In [ ]:
2️⃣3️⃣ How to add timeout?

Answer:

session.get(url, timeout=3)

In [ ]:
2️⃣4️⃣ How to retry async requests?

Answer:
Use loop + await asyncio.sleep(backoff).

In [ ]:
3️⃣1️⃣ What is Semaphore?

Answer:
Limits how many tasks run at the same time.

In [35]:
import asyncio

async def task(name):
    print(f"{name} started")
    await asyncio.sleep(2)
    print(f"{name} finished")

async def main():
    t1 = asyncio.create_task(task("Task A"))
    t2 = asyncio.create_task(task("Task B"))

    await t1
    await t2

await main()


Task A started
Task B started
Task A finished
Task B finished
